# 台灣股市新聞情緒與未來報酬分析

## 專案概述

本 Notebook 針對台灣股市新聞情緒分析系統所收集的資料，進行探索性資料分析（Exploratory Data Analysis, EDA）與統計檢定，主要探討以下問題：

1. **新聞情緒分數**與**未來股票報酬**之間是否存在顯著相關性？
2. **情緒分類**（正面 / 中性 / 負面）能否預測報酬方向？
3. **信心分數**高低是否影響情緒訊號的預測準確度？
4. **日度情緒指標**與股價走勢之間的時序關係為何？

### 資料說明
- **資料期間**：2026 年 4 月 ~ 2026 年 6 月
- **涵蓋標的**：台灣 ETF 及個股（含 0050、2330、2454 等 40+ 檔）
- **情緒標記**：由大型語言模型（LLM）對每則新聞進行情緒分析後，對應至特定標的
- **樣本數**：已對齊報酬資料之新聞 625 筆

### 研究限制說明
> **重要提醒**：本資料集收集期間僅約 2 個月，樣本數量相對有限，統計分析結果可能受到樣本偏差影響，請以審慎的學術態度解讀結果。

## Cell 1：環境設定

載入所需套件，設定中文字型以確保圖表中文顯示正常，並設定資料庫連線路徑。

In [ ]:
import sys
import os

sys.path.insert(0, '../backend')
os.environ['PYTHONIOENCODING'] = 'utf-8'

import sqlite3
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats
import warnings

warnings.filterwarnings('ignore')

# 設定中文字型（Windows 環境優先使用微軟正黑體）
matplotlib.rcParams['font.family'] = ['Microsoft JhengHei', 'Arial Unicode MS', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False
matplotlib.rcParams['figure.dpi'] = 100
matplotlib.rcParams['figure.figsize'] = (12, 6)

# 資料庫路徑
DB_PATH = '../backend/data/twstock_sentiment.db'

print('環境設定完成')
print(f'Python 版本：{sys.version}')
print(f'Pandas 版本：{pd.__version__}')
print(f'NumPy 版本：{np.__version__}')
print(f'資料庫路徑：{os.path.abspath(DB_PATH)}')
print(f'資料庫存在：{os.path.exists(DB_PATH)}')

## Cell 2：讀取資料

從 SQLite 資料庫載入以下資料表：
- `aligned_news_returns`：已對齊報酬之新聞情緒資料（625 筆）
- `llm_news_analysis`：LLM 分析結果（含信心分數、情緒分類等）
- `daily_sentiment`：日度情緒彙整指標
- `stock_prices`：股價資料

In [ ]:
conn = sqlite3.connect(DB_PATH)

# 讀取已對齊報酬之新聞資料，並 JOIN LLM 分析結果
query_aligned = '''
SELECT
    a.news_id,
    a.trading_date,
    a.target,
    a.news_type,
    a.sentiment_score,
    a.future_return_1d,
    a.future_return_3d,
    a.future_return_5d,
    l.sentiment,
    l.confidence,
    l.target_name,
    l.reason
FROM aligned_news_returns a
LEFT JOIN llm_news_analysis l ON a.news_id = l.news_id
ORDER BY a.trading_date
'''
df = pd.read_sql_query(query_aligned, conn)
df['trading_date'] = pd.to_datetime(df['trading_date'])

# 讀取日度情緒資料
df_daily = pd.read_sql_query('SELECT * FROM daily_sentiment ORDER BY trading_date', conn)
df_daily['trading_date'] = pd.to_datetime(df_daily['trading_date'])

# 讀取股價資料（只取 2026 年後的資料）
df_prices = pd.read_sql_query(
    "SELECT stock_id, date, close, open, high, low, volume FROM stock_prices WHERE date >= '2026-04-01' ORDER BY date",
    conn
)
df_prices['date'] = pd.to_datetime(df_prices['date'])

conn.close()

print(f'已讀取 aligned_news_returns（含 LLM 資訊）：{len(df)} 筆')
print(f'已讀取 daily_sentiment：{len(df_daily)} 筆')
print(f'已讀取 stock_prices（2026年後）：{len(df_prices)} 筆')
print()
print('=== aligned 資料前 5 筆預覽 ===')
df.head()

## Cell 3：描述性統計

透過描述性統計，了解資料集的基本特性，包括：
- 樣本量、時間範圍、涵蓋標的數
- 情緒分數分佈
- 新聞類型組成
- 各標的樣本數

In [ ]:
# --- 基本統計摘要 ---
print('=' * 55)
print('   資料集基本統計摘要')
print('=' * 55)
print(f'  總樣本數（已對齊報酬）：{len(df):,} 筆')
print(f'  涵蓋交易日：{df["trading_date"].min().date()} ~ {df["trading_date"].max().date()}')
print(f'  涵蓋標的數：{df["target"].nunique()} 檔')
print(f'  有效 future_return_1d：{df["future_return_1d"].notna().sum():,} 筆')
print(f'  有效 future_return_3d：{df["future_return_3d"].notna().sum():,} 筆')
print(f'  有效 future_return_5d：{df["future_return_5d"].notna().sum():,} 筆')
print()
print('  情緒分數統計：')
print(f'    平均值：{df["sentiment_score"].mean():.4f}')
print(f'    標準差：{df["sentiment_score"].std():.4f}')
print(f'    最小值：{df["sentiment_score"].min():.2f}')
print(f'    最大值：{df["sentiment_score"].max():.2f}')
print('=' * 55)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('資料集描述性統計', fontsize=16, fontweight='bold', y=1.01)

# --- 子圖 1：情緒分數分佈（直方圖）---
ax1 = axes[0, 0]
ax1.hist(df['sentiment_score'].dropna(), bins=25, color='steelblue', edgecolor='white', linewidth=0.5)
ax1.axvline(df['sentiment_score'].mean(), color='red', linestyle='--', linewidth=1.5,
            label=f'平均值 = {df["sentiment_score"].mean():.3f}')
ax1.set_title('情緒分數分佈', fontsize=13)
ax1.set_xlabel('情緒分數', fontsize=11)
ax1.set_ylabel('筆數', fontsize=11)
ax1.legend(fontsize=10)
ax1.grid(axis='y', alpha=0.4)

# --- 子圖 2：新聞類型分佈（長條圖）---
ax2 = axes[0, 1]
type_counts = df['news_type'].value_counts()
type_labels_map = {'market': '市場新聞\n(market)', 'stock': '個股新聞\n(stock)',
                   'industry': '產業新聞\n(industry)', 'etf': 'ETF 新聞\n(etf)'}
type_labels = [type_labels_map.get(k, k) for k in type_counts.index]
bars = ax2.bar(type_labels, type_counts.values,
               color=['#2196F3', '#4CAF50', '#FF9800', '#9C27B0'])
for bar, val in zip(bars, type_counts.values):
    ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 3,
             f'{val}\n({val/len(df)*100:.1f}%)', ha='center', va='bottom', fontsize=9)
ax2.set_title('新聞類型分佈', fontsize=13)
ax2.set_ylabel('筆數', fontsize=11)
ax2.grid(axis='y', alpha=0.4)
ax2.set_ylim(0, type_counts.max() * 1.2)

# --- 子圖 3：情緒分類分佈（圓餅圖）---
ax3 = axes[1, 0]
sentiment_counts = df['sentiment'].value_counts()
sentiment_colors = {'positive': '#4CAF50', 'neutral': '#9E9E9E', 'negative': '#F44336'}
colors = [sentiment_colors.get(k, '#9E9E9E') for k in sentiment_counts.index]
label_map = {'positive': '正面', 'neutral': '中性', 'negative': '負面'}
labels = [f"{label_map.get(k, k)}\n({v}筆, {v/len(df)*100:.1f}%)"
          for k, v in zip(sentiment_counts.index, sentiment_counts.values)]
ax3.pie(sentiment_counts.values, labels=labels, colors=colors,
        autopct=None, startangle=90, pctdistance=0.75,
        wedgeprops=dict(edgecolor='white', linewidth=1.5))
ax3.set_title('情緒分類分佈', fontsize=13)

# --- 子圖 4：前 10 大標的樣本數（水平長條圖）---
ax4 = axes[1, 1]
top_targets = df['target'].value_counts().head(10)
bars4 = ax4.barh(top_targets.index[::-1], top_targets.values[::-1],
                 color='#42A5F5')
for bar, val in zip(bars4, top_targets.values[::-1]):
    ax4.text(val + 1, bar.get_y() + bar.get_height() / 2,
             f'{val}', va='center', fontsize=10)
ax4.set_title('前 10 大標的樣本數', fontsize=13)
ax4.set_xlabel('樣本數（筆）', fontsize=11)
ax4.set_ylabel('標的代碼', fontsize=11)
ax4.grid(axis='x', alpha=0.4)
ax4.set_xlim(0, top_targets.max() * 1.15)

plt.tight_layout()
plt.show()
print('\n說明：市場新聞（market）佔最大比例，因其可同時對應多個標的。')
print('      正面情緒新聞數量略多於中性與負面，整體偏正面。')

## Cell 4：情緒分數與未來報酬相關性分析

本節計算情緒分數（`sentiment_score`，範圍 -1 ~ +1）與未來 1 日、3 日、5 日報酬（`future_return_1d/3d/5d`）之間的 **Pearson 相關係數**，並繪製散佈圖加以視覺化。

此外，進一步按標的分組（0050、2330、2454），分別計算相關係數，觀察是否具有異質性。

> **統計解讀注意事項**：由於樣本僅約 2 個月，相關係數即使統計顯著，實際預測能力可能仍十分有限，需避免過度詮釋。

In [ ]:
# 計算整體相關性
return_cols = ['future_return_1d', 'future_return_3d', 'future_return_5d']
return_labels = ['未來 1 日報酬', '未來 3 日報酬', '未來 5 日報酬']

print('=' * 65)
print('   情緒分數 vs 未來報酬：Pearson 相關係數（全樣本）')
print('=' * 65)
print(f'{"期間":<12} {"相關係數":>10} {"p 值":>12} {"有效樣本":>10} {"顯著性":>8}')
print('-' * 65)

corr_results = {}
for col, label in zip(return_cols, return_labels):
    mask = df['sentiment_score'].notna() & df[col].notna()
    x = df.loc[mask, 'sentiment_score']
    y = df.loc[mask, col]
    r, p = stats.pearsonr(x, y)
    n = mask.sum()
    sig = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'n.s.'))
    corr_results[col] = {'r': r, 'p': p, 'n': n, 'sig': sig}
    print(f'{label:<12} {r:>10.4f} {p:>12.4f} {n:>10} {sig:>8}')

print('-' * 65)
print('顯著性：*** p<0.001  ** p<0.01  * p<0.05  n.s. 不顯著')
print()
print('解讀說明：')
print('  相關係數接近 0 表示情緒分數與未來報酬線性關係微弱。')
print('  在短樣本期間內，此結果符合市場效率假說的預期。')

In [ ]:
# 散佈圖：情緒分數 vs 未來 1d/3d/5d 報酬
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('情緒分數 vs 未來報酬散佈圖（全樣本）', fontsize=14, fontweight='bold')

colors_by_sentiment = df['sentiment'].map({'positive': '#4CAF50', 'neutral': '#9E9E9E', 'negative': '#F44336'})
colors_by_sentiment = colors_by_sentiment.fillna('#9E9E9E')

for i, (col, label) in enumerate(zip(return_cols, return_labels)):
    ax = axes[i]
    mask = df['sentiment_score'].notna() & df[col].notna()
    x = df.loc[mask, 'sentiment_score']
    y = df.loc[mask, col]
    c = colors_by_sentiment[mask]

    ax.scatter(x, y, c=c, alpha=0.4, s=25, edgecolors='none')

    # 趨勢線
    if len(x) > 2:
        z = np.polyfit(x, y, 1)
        p_line = np.poly1d(z)
        x_range = np.linspace(x.min(), x.max(), 100)
        ax.plot(x_range, p_line(x_range), 'r--', linewidth=1.5, alpha=0.8)

    r = corr_results[col]['r']
    p_val = corr_results[col]['p']
    sig = corr_results[col]['sig']
    ax.set_title(f'{label}\nr = {r:.4f}, p = {p_val:.4f} ({sig})', fontsize=11)
    ax.set_xlabel('情緒分數', fontsize=10)
    ax.set_ylabel('報酬率', fontsize=10)
    ax.axhline(0, color='black', linewidth=0.8, linestyle=':')
    ax.axvline(0, color='black', linewidth=0.8, linestyle=':')
    ax.grid(alpha=0.3)

# 圖例說明
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#4CAF50', label='正面情緒'),
    Patch(facecolor='#9E9E9E', label='中性情緒'),
    Patch(facecolor='#F44336', label='負面情緒')
]
fig.legend(handles=legend_elements, loc='lower center', ncol=3,
           bbox_to_anchor=(0.5, -0.06), fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
# 按主要標的分組計算相關係數
main_targets = ['0050', '2330', '2454']
target_names = {'0050': '0050 元大台灣50', '2330': '2330 台積電', '2454': '2454 聯發科'}

print('=' * 75)
print('   依標的分組：情緒分數 vs 未來報酬 Pearson 相關係數')
print('=' * 75)

all_corr_data = []

for tgt in main_targets:
    df_tgt = df[df['target'] == tgt]
    name = target_names.get(tgt, tgt)
    print(f'\n  [{name}]  樣本數：{len(df_tgt)} 筆')
    print(f'  {"期間":<12} {"相關係數":>10} {"p 值":>12} {"有效樣本":>10} {"顯著性":>8}')
    print('  ' + '-' * 55)
    for col, label in zip(return_cols, return_labels):
        mask = df_tgt['sentiment_score'].notna() & df_tgt[col].notna()
        if mask.sum() < 5:
            print(f'  {label:<12} {"-- 樣本不足 --":>32}')
            continue
        x = df_tgt.loc[mask, 'sentiment_score']
        y = df_tgt.loc[mask, col]
        r, p = stats.pearsonr(x, y)
        n = mask.sum()
        sig = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'n.s.'))
        print(f'  {label:<12} {r:>10.4f} {p:>12.4f} {n:>10} {sig:>8}')
        all_corr_data.append({'標的': name, '期間': label, '相關係數': r, 'p值': p, '樣本數': n})

print('\n顯著性：*** p<0.001  ** p<0.01  * p<0.05  n.s. 不顯著')

## Cell 5：情緒分類 vs 報酬方向分析

將情緒分類（正面 / 中性 / 負面）與報酬方向（正報酬 / 負報酬）進行交叉列表分析，計算各情緒類別下報酬為正的比例，並使用 **Chi-square 檢定**評估兩者之間的獨立性。

> **說明**：若情緒分類具有預測能力，則正面情緒後的正報酬比例應顯著高於負面情緒後。

In [ ]:
# 建立報酬方向欄位（以 future_return_1d 為主）
df_valid = df[df['future_return_1d'].notna() & df['sentiment'].notna()].copy()
df_valid['return_direction'] = (df_valid['future_return_1d'] > 0).astype(int)
df_valid['return_label'] = df_valid['return_direction'].map({1: '正報酬', 0: '負報酬/持平'})

# 僅保留標準情緒分類
df_valid = df_valid[df_valid['sentiment'].isin(['positive', 'neutral', 'negative'])]

sentiment_order = ['positive', 'neutral', 'negative']
sentiment_cn = {'positive': '正面情緒', 'neutral': '中性情緒', 'negative': '負面情緒'}

# 交叉列表
ct = pd.crosstab(df_valid['sentiment'], df_valid['return_label'])
ct_pct = ct.div(ct.sum(axis=1), axis=0) * 100

# 確保 index 順序
ct = ct.reindex([s for s in sentiment_order if s in ct.index])
ct_pct = ct_pct.reindex([s for s in sentiment_order if s in ct_pct.index])

ct.index = [sentiment_cn.get(s, s) for s in ct.index]
ct_pct.index = [sentiment_cn.get(s, s) for s in ct_pct.index]

print('交叉列表（原始筆數）：')
print(ct)
print('\n交叉列表（列百分比 %）：')
print(ct_pct.round(2))

# Chi-square 檢定
ct_raw = pd.crosstab(df_valid['sentiment'], df_valid['return_label'])
ct_raw = ct_raw.reindex([s for s in sentiment_order if s in ct_raw.index])
chi2, p_chi, dof, expected = stats.chi2_contingency(ct_raw)
print(f'\nChi-square 檢定結果：')
print(f'  Chi2 = {chi2:.4f},  自由度 = {dof},  p 值 = {p_chi:.4f}')
if p_chi < 0.05:
    print('  → 情緒分類與報酬方向之間的關聯「統計顯著」（p < 0.05）')
else:
    print('  → 情緒分類與報酬方向之間的關聯「不顯著」（p >= 0.05）')
    print('  → 即：情緒分類無法顯著預測短期報酬方向，此為常見結果')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('情緒分類 vs 報酬方向分析（以 1 日未來報酬為基準）',
             fontsize=14, fontweight='bold')

# --- 子圖 1：堆疊長條圖 ---
ax1 = axes[0]
labels_order = [sentiment_cn[s] for s in sentiment_order if s in [df_valid['sentiment'].unique()]]
labels_order = [sentiment_cn[s] for s in sentiment_order if s in df_valid['sentiment'].values]

pos_pct = []
neg_pct = []
for s in sentiment_order:
    if s not in df_valid['sentiment'].values:
        continue
    grp = df_valid[df_valid['sentiment'] == s]
    pos = (grp['return_direction'] == 1).sum() / len(grp) * 100
    neg = 100 - pos
    pos_pct.append(pos)
    neg_pct.append(neg)

x = np.arange(len(labels_order))
b1 = ax1.bar(x, pos_pct, label='正報酬', color='#4CAF50', alpha=0.85)
b2 = ax1.bar(x, neg_pct, bottom=pos_pct, label='負報酬/持平', color='#F44336', alpha=0.85)
ax1.axhline(50, color='black', linestyle='--', linewidth=1, alpha=0.7, label='50% 基準線')

for bar, pct in zip(b1, pos_pct):
    ax1.text(bar.get_x() + bar.get_width() / 2, pct / 2,
             f'{pct:.1f}%', ha='center', va='center', fontsize=11, color='white', fontweight='bold')

ax1.set_xticks(x)
ax1.set_xticklabels(labels_order, fontsize=11)
ax1.set_ylabel('比例（%）', fontsize=11)
ax1.set_title('各情緒類別之正/負報酬比例', fontsize=12)
ax1.legend(loc='upper right', fontsize=10)
ax1.set_ylim(0, 115)
ax1.grid(axis='y', alpha=0.4)

# --- 子圖 2：各情緒分類平均 1 日報酬（箱型圖）---
ax2 = axes[1]
box_data = [df_valid[df_valid['sentiment'] == s]['future_return_1d'].dropna().values
            for s in sentiment_order if s in df_valid['sentiment'].values]
bp = ax2.boxplot(box_data, labels=labels_order, patch_artist=True,
                 medianprops=dict(color='black', linewidth=1.5),
                 whiskerprops=dict(linewidth=1),
                 capprops=dict(linewidth=1))

box_colors = ['#4CAF50', '#9E9E9E', '#F44336']
for patch, color in zip(bp['boxes'], box_colors[:len(bp['boxes'])]):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax2.axhline(0, color='black', linestyle='--', linewidth=1, alpha=0.7)
ax2.set_ylabel('未來 1 日報酬率', fontsize=11)
ax2.set_title(f'各情緒分類之 1 日報酬分佈\n(Chi2={chi2:.3f}, p={p_chi:.3f})', fontsize=12)
ax2.grid(axis='y', alpha=0.4)

# 標記平均值
for i, s in enumerate([s for s in sentiment_order if s in df_valid['sentiment'].values]):
    mean_val = df_valid[df_valid['sentiment'] == s]['future_return_1d'].mean()
    ax2.scatter(i + 1, mean_val, color='gold', s=80, zorder=5, marker='D',
                edgecolors='black', linewidths=0.8)

plt.tight_layout()
plt.show()

print('\n各情緒分類之平均報酬率：')
for s in sentiment_order:
    if s not in df_valid['sentiment'].values:
        continue
    mean_r = df_valid[df_valid['sentiment'] == s]['future_return_1d'].mean()
    n = df_valid[df_valid['sentiment'] == s]['future_return_1d'].notna().sum()
    print(f'  {sentiment_cn[s]}：平均 {mean_r*100:.4f}%  (n={n})')

## Cell 6：時間序列分析

以 **0050（元大台灣 50 ETF）** 為例，繪製日度情緒平均值（`sentiment_avg`）與股價收盤價的時序走勢圖，觀察兩者是否具有同步變動或領先/落後關係。

> **說明**：若情緒指標具有前瞻性，理論上情緒轉折點應領先股價數日。在視覺分析中，我們也納入 3 日移動平均情緒（`sentiment_ma3`）以平滑雜訊。

In [ ]:
# 準備 0050 的情緒與股價資料
ds_0050 = df_daily[df_daily['target'] == '0050'].copy()
ds_0050 = ds_0050.set_index('trading_date').sort_index()

price_0050 = df_prices[df_prices['stock_id'] == '0050'].copy()
price_0050 = price_0050.set_index('date').sort_index()

print(f'0050 日度情緒資料：{len(ds_0050)} 個交易日')
print(f'0050 股價資料（2026年後）：{len(price_0050)} 個交易日')
print()

# 確認日期範圍重疊
common_dates = ds_0050.index.intersection(price_0050.index)
print(f'情緒與股價共同交易日：{len(common_dates)} 天')
print(f'起訖：{common_dates.min().date()} ~ {common_dates.max().date()}')

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 9), sharex=True)
fig.suptitle('0050 元大台灣50：日度情緒指標 vs 股價走勢', fontsize=14, fontweight='bold')

# --- 上方子圖：情緒指標 ---
ax1.fill_between(ds_0050.index, ds_0050['sentiment_avg'], 0,
                 where=ds_0050['sentiment_avg'] >= 0,
                 color='#4CAF50', alpha=0.4, label='正面情緒區間')
ax1.fill_between(ds_0050.index, ds_0050['sentiment_avg'], 0,
                 where=ds_0050['sentiment_avg'] < 0,
                 color='#F44336', alpha=0.4, label='負面情緒區間')
ax1.plot(ds_0050.index, ds_0050['sentiment_avg'],
         color='#1565C0', linewidth=1.2, alpha=0.8, label='日度情緒平均')

if ds_0050['sentiment_ma3'].notna().sum() > 0:
    ax1.plot(ds_0050.index, ds_0050['sentiment_ma3'],
             color='#FF6F00', linewidth=1.5, linestyle='--', alpha=0.9, label='情緒 MA3')

ax1.axhline(0, color='black', linewidth=0.8, linestyle='-')
ax1.set_ylabel('情緒分數', fontsize=11)
ax1.legend(loc='upper left', fontsize=9)
ax1.grid(alpha=0.3)
ax1.set_ylim(-1.2, 1.2)

# 標記新聞筆數
if 'news_count' in ds_0050.columns:
    ax1_twin = ax1.twinx()
    ax1_twin.bar(ds_0050.index, ds_0050['news_count'],
                 alpha=0.15, color='gray', width=0.8, label='新聞筆數')
    ax1_twin.set_ylabel('新聞筆數', fontsize=10, color='gray')
    ax1_twin.tick_params(axis='y', labelcolor='gray')

# --- 下方子圖：股價 ---
price_plot = price_0050.reindex(ds_0050.index).dropna()
if len(price_plot) > 0:
    ax2.plot(price_plot.index, price_plot['close'],
             color='#1565C0', linewidth=2, label='0050 收盤價')
    ax2.fill_between(price_plot.index, price_plot['close'].min() * 0.99,
                     price_plot['close'], alpha=0.1, color='#1565C0')

    # 標記情緒特別高/低的日期
    high_sent = ds_0050[ds_0050['sentiment_avg'] > 0.7].index
    low_sent = ds_0050[ds_0050['sentiment_avg'] < -0.5].index

    high_prices = price_plot.reindex(high_sent).dropna()
    low_prices = price_plot.reindex(low_sent).dropna()

    if len(high_prices) > 0:
        ax2.scatter(high_prices.index, high_prices['close'],
                    color='#4CAF50', s=80, zorder=5, label='高正向情緒日', marker='^')
    if len(low_prices) > 0:
        ax2.scatter(low_prices.index, low_prices['close'],
                    color='#F44336', s=80, zorder=5, label='高負向情緒日', marker='v')

ax2.set_ylabel('股價（NTD）', fontsize=11)
ax2.set_xlabel('日期', fontsize=11)
ax2.legend(loc='upper left', fontsize=9)
ax2.grid(alpha=0.3)

plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

# 計算情緒與價格變化的簡單相關
if len(common_dates) > 5:
    aligned_sentiment = ds_0050.reindex(common_dates)['sentiment_avg']
    aligned_price = price_0050.reindex(common_dates)['close']
    aligned_price_return = aligned_price.pct_change()

    mask = aligned_sentiment.notna() & aligned_price_return.notna()
    if mask.sum() > 5:
        r_ts, p_ts = stats.pearsonr(aligned_sentiment[mask], aligned_price_return[mask])
        print(f'\n0050 日度情緒平均 vs 同日股價報酬率：r = {r_ts:.4f}, p = {p_ts:.4f}')
        print('（此為同期相關，非預測性關係）')

## Cell 7：信心分數分析

LLM 在進行情緒分析時，同時輸出一個 `confidence`（信心分數，0 ~ 1）。本節探討：**信心分數越高，情緒訊號是否越準確？**

分析方式：
1. 將樣本依信心分數切為四個分位數（Q1 最低 ~ Q4 最高）
2. 計算各分位數內：情緒方向（正/負）與報酬方向（正/負）的**一致率**（預測準確度）
3. 繪製信心分數分佈與各分位數準確率比較圖

In [ ]:
# 準備信心分數分析資料
df_conf = df[df['confidence'].notna() & df['future_return_1d'].notna() &
             df['sentiment'].isin(['positive', 'negative'])].copy()

# 定義情緒訊號方向與報酬方向的一致性
df_conf['sentiment_dir'] = df_conf['sentiment'].map({'positive': 1, 'negative': -1})
df_conf['return_dir'] = np.sign(df_conf['future_return_1d'])
df_conf['signal_correct'] = (df_conf['sentiment_dir'] == df_conf['return_dir']).astype(int)

print(f'可用樣本（非中性、有報酬資料）：{len(df_conf)} 筆')
print(f'整體準確率：{df_conf["signal_correct"].mean()*100:.1f}%')
print(f'（純隨機猜測基準：約 50%）')
print()

# 切分四個信心分位數
df_conf['conf_quartile'] = pd.qcut(df_conf['confidence'], q=4, labels=['Q1\n(低信心)', 'Q2', 'Q3', 'Q4\n(高信心)'])

quartile_stats = df_conf.groupby('conf_quartile').agg(
    樣本數=('signal_correct', 'count'),
    準確率=('signal_correct', 'mean'),
    信心分數均值=('confidence', 'mean')
).reset_index()
quartile_stats['準確率（%）'] = (quartile_stats['準確率'] * 100).round(2)
quartile_stats['信心分數均值'] = quartile_stats['信心分數均值'].round(4)

print('依信心分數四分位數統計：')
print(quartile_stats[['conf_quartile', '樣本數', '信心分數均值', '準確率（%）']].to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('信心分數分析', fontsize=14, fontweight='bold')

# --- 子圖 1：信心分數分佈直方圖 ---
ax1 = axes[0]
ax1.hist(df['confidence'].dropna(), bins=20, color='#42A5F5',
         edgecolor='white', linewidth=0.5)
ax1.axvline(df['confidence'].mean(), color='red', linestyle='--',
            linewidth=1.5, label=f'平均值 = {df["confidence"].mean():.3f}')
ax1.set_title('信心分數分佈', fontsize=12)
ax1.set_xlabel('信心分數', fontsize=11)
ax1.set_ylabel('筆數', fontsize=11)
ax1.legend(fontsize=10)
ax1.grid(axis='y', alpha=0.4)

# --- 子圖 2：各信心分位數準確率 ---
ax2 = axes[1]
bar_colors = ['#EF9A9A', '#FFCC80', '#A5D6A7', '#42A5F5']
bars = ax2.bar(quartile_stats['conf_quartile'].astype(str),
               quartile_stats['準確率（%）'],
               color=bar_colors[:len(quartile_stats)],
               edgecolor='white', linewidth=0.5)
ax2.axhline(50, color='black', linestyle='--', linewidth=1.5,
            label='隨機基準線 (50%)', alpha=0.8)
overall_acc = df_conf['signal_correct'].mean() * 100
ax2.axhline(overall_acc, color='red', linestyle=':', linewidth=1.5,
            label=f'整體準確率 ({overall_acc:.1f}%)', alpha=0.8)

for bar, val, n in zip(bars, quartile_stats['準確率（%）'], quartile_stats['樣本數']):
    ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
             f'{val:.1f}%\n(n={n})', ha='center', va='bottom', fontsize=10)

ax2.set_title('依信心分位數：情緒方向 vs 報酬方向一致率', fontsize=11)
ax2.set_xlabel('信心分位數', fontsize=11)
ax2.set_ylabel('準確率（%）', fontsize=11)
ax2.legend(fontsize=10)
ax2.set_ylim(0, max(quartile_stats['準確率（%）'].max() + 15, 65))
ax2.grid(axis='y', alpha=0.4)

plt.tight_layout()
plt.show()

print(f'\n解讀說明：')
print(f'  整體情緒方向與報酬方向的一致率為 {overall_acc:.1f}%，')
if overall_acc > 52:
    print(f'  略高於隨機猜測（50%），但差異較小，不宜單獨作為交易訊號。')
else:
    print(f'  接近或低於隨機猜測（50%），顯示 LLM 情緒訊號在短期預測上效果有限。')
print(f'  建議與其他技術指標或基本面訊號配合使用。')

## Cell 8：研究小結與學術反思

本節整理所有分析結果，以誠實的學術語言呈現發現、局限性及未來改進方向。

In [ ]:
# 彙整統計摘要
print('=' * 60)
print('   研究結果摘要')
print('=' * 60)

print()
print('【資料集概況】')
print(f'  樣本數：{len(df)} 筆新聞-報酬配對')
print(f'  涵蓋期間：{df["trading_date"].min().date()} ~ {df["trading_date"].max().date()}')
print(f'  涵蓋標的：{df["target"].nunique()} 檔股票/ETF')
print(f'  主要標的：0050 (n={len(df[df["target"]=="0050"])}), '
      f'2330 (n={len(df[df["target"]=="2330"])}), '
      f'2454 (n={len(df[df["target"]=="2454"])})')

print()
print('【主要發現】')

# 相關係數
mask_1d = df['sentiment_score'].notna() & df['future_return_1d'].notna()
r_1d, p_1d = stats.pearsonr(df.loc[mask_1d, 'sentiment_score'], df.loc[mask_1d, 'future_return_1d'])
mask_5d = df['sentiment_score'].notna() & df['future_return_5d'].notna()
r_5d, p_5d = stats.pearsonr(df.loc[mask_5d, 'sentiment_score'], df.loc[mask_5d, 'future_return_5d'])

print(f'  1. 情緒分數與未來 1 日報酬 Pearson r = {r_1d:.4f}（p = {p_1d:.4f}）')
print(f'     情緒分數與未來 5 日報酬 Pearson r = {r_5d:.4f}（p = {p_5d:.4f}）')
print(f'     → 整體相關性弱，符合弱式效率市場假說的預期')

print(f'  2. Chi-square 檢定：Chi2 = {chi2:.4f}, p = {p_chi:.4f}')
if p_chi < 0.05:
    print(f'     → 情緒分類與報酬方向存在統計顯著關聯')
else:
    print(f'     → 情緒分類對報酬方向無顯著預測力')

if 'overall_acc' in dir():
    print(f'  3. 情緒方向預測準確率：{overall_acc:.1f}%（隨機基準：50%）')

print()
print('【研究局限性】')
print('  1. 樣本期間過短（約 2 個月），可能無法代表長期市場行為')
print('  2. 部分標的樣本數極少（如 2454 僅 49 筆），統計推論可靠性低')
print('  3. LLM 情緒分析本身可能含有系統性偏差（如過度正面傾向）')
print('  4. 未控制市場整體走勢、產業效應等混淆因素')
print('  5. 短期報酬（1~5 日）受雜訊影響大，情緒訊號的效果可能被稀釋')

print()
print('【未來研究方向】')
print('  1. 延長資料收集期間至 1 年以上，增加樣本數')
print('  2. 使用更長時間窗口的報酬（20~60 日），可能更能捕捉情緒影響')
print('  3. 加入交易量、新聞數量等調節變數，建立多因子模型')
print('  4. 與傳統技術指標（如 RSI、MACD）進行比較研究')
print('  5. 探討情緒訊號的非線性關係（極端情緒 vs 溫和情緒的差異）')
print('=' * 60)

## 研究小結

### 核心發現

本研究基於約 2 個月的台灣股市新聞資料（625 筆情緒-報酬配對），針對 LLM 情緒分析與未來股價報酬的關係進行探索性分析，主要發現如下：

**1. 情緒分數與報酬的線性相關性極弱**

全樣本 Pearson 相關係數絕對值均低於 0.1，且統計顯著性不穩定。這符合**弱式效率市場假說**（Fama, 1970）的預期：若市場能快速消化公開資訊（包含新聞情緒），則情緒訊號應無法持續預測未來報酬。

**2. 情緒分類對報酬方向的預測力有限**

Chi-square 檢定結果顯示，情緒分類（正面/中性/負面）與 1 日後報酬方向之間的統計關聯不顯著，各情緒類別下的正報酬比例差異不大，接近 50% 的隨機基準。

**3. 信心分數分析**

LLM 輸出的信心分數平均為 0.78，整體偏高。不同信心分位數下的情緒預測準確率差異有限，尚未觀察到「高信心 = 高準確率」的一致規律。

**4. 時序關係視覺化**

0050 的日度情緒指標與股價走勢在視覺上有部分對應關係，但因樣本期間較短，難以進行嚴格的格蘭傑因果（Granger causality）檢定。

### 學術誠信聲明

> 本研究在設計上具有**探索性**而非**驗證性**性質。鑑於樣本期間僅約 2 個月，分析結果的外部效度（external validity）有限，不宜據此作出強烈的投資建議或理論主張。結果的誠實呈現本身即具有學術價值——**陰性結果**（null results）同樣是重要的科學知識。

### 參考文獻方向

- Fama, E. F. (1970). Efficient capital markets: A review of theory and empirical work. *Journal of Finance*, 25(2), 383–417.
- Tetlock, P. C. (2007). Giving content to investor sentiment: The role of media in the stock market. *Journal of Finance*, 62(3), 1139–1168.
- Bollen, J., Mao, H., & Zeng, X. (2011). Twitter mood predicts the stock market. *Journal of Computational Science*, 2(1), 1–8.
- Lopez-Lira, T., & Tang, Y. (2023). Can ChatGPT forecast stock price movements? *arXiv preprint*, arXiv:2304.07619.